# DAPO vs. Base

 Qwen3-1.7B-Base with DAPO + LoRA: 20 steps, four accepted groups per step, four replies generated together per question, 2048-token training/evaluation limits, and 50 fixed evaluation questions. No automatic training or experiment deadline. This is a small experiment, not an exact paper reproduction; accuracy gains are not guaranteed.
Training data: 512 seeded MATH train Level 1-3 problems. A separate pre-training cell measures base accuracy on 10 held-out MATH500 questions. Final comparison still uses 50 questions.

In [7]:
from google.colab import drive
from pathlib import Path
import shutil, subprocess, sys, os, time

drive.mount("/content/drive")
SOURCE = Path("/content/drive/MyDrive/dapo-base-compare")
PROJECT = Path("/content/dapo-base-compare-clean")
assert (SOURCE / "setup_colab.py").exists(), "Upload the complete NEW project folder, including setup_colab.py"
shutil.copytree(SOURCE, PROJECT, dirs_exist_ok=True)
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
ENV = os.environ.copy()
ENV.pop("PYTHONPATH", None)
ENV.pop("PYTHONHOME", None)
ENV["PYTHONNOUSERSITE"] = "1"
subprocess.run(["nvidia-smi"], check=True)
print("Project copied to", PROJECT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project copied to /content/dapo-base-compare-clean


Install and check the isolated environment.

In [8]:
subprocess.run([sys.executable, "setup_colab.py"], cwd=PROJECT, env=ENV, check=True)
PYTHON = "/content/dapo-venv/bin/python"
assert Path(PYTHON).exists()
print("Use this interpreter for all project scripts:", PYTHON)


Use this interpreter for all project scripts: /content/dapo-venv/bin/python


Before training: evaluate the original base on 10 fixed held-out MATH500 questions. This is a small diagnostic, not a reliable benchmark estimate or training-set accuracy. No LoRA adapter is loaded. Results and full answers are saved to Drive. Run this cell after setup and before the training cell.

In [9]:
import json
from datetime import datetime
from live_process import run_live
BASE_CHECK = PROJECT / "results" / ("base-check-" + datetime.now().strftime("%Y%m%d-%H%M%S"))
BASE_CHECK.mkdir(parents=True, exist_ok=True)
check_cfg = json.loads((PROJECT / "config.json").read_text())
check_cfg["eval_samples"] = 1
check_cfg["eval_max_tokens"] = 2048
check_config = BASE_CHECK / "config.json"
check_config.write_text(json.dumps(check_cfg, indent=2), encoding="utf-8")
run_live([PYTHON, "-u", "prepare.py", "--out", str(BASE_CHECK / "data"),
          "--train-size", "512", "--eval-size", "50", "--seed", str(check_cfg["seed"])],
         cwd=PROJECT, env=ENV, log_path=BASE_CHECK / "console.log", label="Prepare MATH data")
run_live([PYTHON, "-u", "evaluate.py", "--config", str(check_config),
          "--data", str(BASE_CHECK / "data/eval.jsonl"), "--limit", "10",
          "--out", str(BASE_CHECK / "base")],
         cwd=PROJECT, env=ENV, log_path=BASE_CHECK / "console.log", label="Base check: 10 questions")
summary = json.loads((BASE_CHECK / "base/summary.json").read_text())
print(f"Base accuracy on {summary['n']} questions: {summary['accuracy']:.2%}")
print(f"Truncation rate: {summary['truncation_rate']:.2%}")
print("Full answers:", BASE_CHECK / "base/predictions.jsonl")
destination = Path("/content/drive/MyDrive/dapo-experiments") / BASE_CHECK.name
shutil.copytree(BASE_CHECK, destination, dirs_exist_ok=True)
print("Saved:", destination)


[START] Prepare MATH data | PID=11730
[DATA] Resolving dataset revisions and downloading evaluation data

Generating train split: 0 examples [00:00, ? examples/s]
Generating train split: 7500 examples [00:00, 179908.04 examples/s]
[DATA] MATH train Level 1-3: eligible=3501 selected=512
Saved 512 train rows and 50 evaluation rows to /content/dapo-base-compare-clean/results/base-check-20261011-222207/data
[END] Prepare MATH data | exit=0 | elapsed=0.2 min
[START] Base check: 10 questions | PID=11819
[MODEL] Loading original base for 10 evaluation questions
`torch_dtype` is deprecated! Use `dtype` instead!
[EVAL] Model loaded. Starting generation.
1/10 accuracy=1.000
[HEARTBEAT] Base check: 10 questions | elapsed=0.5 min
2/10 accuracy=0.500
3/10 accuracy=0.333
[HEARTBEAT] Base check: 10 questions | elapsed=1.0 min
4/10 accuracy=0.250
5/10 accuracy=0.200
6/10 accuracy=0.333
[HEARTBEAT] Base check: 10 questions | elapsed=1.5 min
7/10 accuracy=0.286
8/10 accuracy=0.375
9/10 accuracy=0.333
[H

Run the experiment to completion. Setup has checked dependencies and CUDA. Training uses the configured number of steps, then both models are evaluated on the same questions. There is no 45-minute or 105-minute cutoff. Elapsed time and live progress remain visible. Runtime depends on sampling acceptance and generation length.

In [10]:
from live_process import run_live
print("MATH-L123-G4-2048-v3: 20 steps; 4 accepted groups/step; 4 batched replies/question; 2048 tokens; no time limit")
result = run_live([PYTHON, "-u", "run_budget.py", "--eval-size", "50"],
                  cwd=PROJECT, env=ENV,
                  log_path=PROJECT / "notebook_console.log", label="DAPO experiment")
print("Runner exit code:", result)


Streaming output truncated to the last 5000 lines.

[RESPONSE 2/4]
\begin{align*}
\triangle+q&=59, \\
(\triangle+q)+q&=106.
\end{align*}Subtracting the first equation from the second equation yields
\begin{align*}
(\triangle+q)+q-[(\triangle+q)&=106-59, \\
q&=47.
\end{align*}Substituting $q=47$ into the first equation yields
\begin{align*}
\triangle+q&=59, \\
\triangle+47&=59, \\
\triangle&=\boxed{12}.
\end{align*}

To solve the system of equations, we start with the given equations:

1. \(\triangle + q = 59\)
2. \((\triangle + q) + q = 106\)

First, substitute the expression for \(\triangle + q\) from the first equation into the second equation. This gives us:
\[
(59) + q = 106
\]
Simplify the left side:
\[
59 + q = 106
\]
Next, solve for \(q\) by subtracting 59 from both sides:
\[
q = 106 - 59
\]
\[
q = 47
\]

Now that we have the value of \(q\), we substitute \(q = 47\) back into the first equation to find \(\triangle\):
\[
\triangle + 47 = 59
\]
Solve for \(\triangle\) by subtracti

Inspect completion status and persist results. If you interrupt the run or Colab disconnects, inspect partial outputs before retrying.

In [11]:
import json
from IPython.display import Markdown, display
statuses = sorted((PROJECT / "runs").glob("budget-*/budget_status.json"), key=lambda p: p.stat().st_mtime)
if statuses:
    status_path = statuses[-1]
    status = json.loads(status_path.read_text())
    print(json.dumps(status, indent=2))
    if status.get("status") == "complete":
        display(Markdown(Path(status["report"]).read_text()))
    destination = Path("/content/drive/MyDrive/dapo-experiments") / status["run"]
    destination.mkdir(parents=True, exist_ok=True)
    for source, name in [(status_path.parent, "runs"), (PROJECT / "results" / status["run"], "results")]:
        if source.exists():
            shutil.copytree(source, destination / name, dirs_exist_ok=True)
    print("Saved:", destination)
else:
    print("No budget status found. Inspect the previous error before retrying.")


{
  "run": "budget-20261011-222450",
  "budget_minutes": null,
  "eval_size": 50,
  "status": "complete",
  "stage": "compare.py",
  "elapsed_minutes": 314.6351919064834,
  "completed_training_steps": 20,
  "report": "results/budget-20261011-222450/comparison/report.md"
}


# Base vs DAPO

Questions: 50

| Model | Accuracy |
|---|---:|
| Base | 26.00% |
| DAPO + LoRA | 34.00% |

Absolute change: **+8.00 percentage points**.

Paired bootstrap 95% interval: [-2.00, +20.00] percentage points.

Improved: 6; regressed: 2; unchanged: 42.

An interval crossing zero is inconclusive. This interval captures question-sampling uncertainty, not variability across independent training seeds. No gain is guaranteed.


Saved: /content/drive/MyDrive/dapo-experiments/budget-20261011-222450
